# Lab 2 · ตามรอย prompt: 3 ระดับ วัดด้วยตัวชี้วัดเดียวกัน
**สัปดาห์ 3 · ช่วงบ่าย (50 นาที)**

โจทย์เดียวกัน (สรุปรีวิวร้านกาแฟเป็น JSON) เขียน prompt 3 ระดับ แล้วตามไปตามเส้นทาง 6 ขั้นของช่วงเช้า

| ระดับ | prompt | มีอะไรบ้าง |
|---|---|---|
| **A · คลุมเครือ** | "สรุปรีวิวนี้หน่อย: …" | ไม่ได้บอกด้วยซ้ำว่าอยากได้ JSON |
| **B · สั้น** | "สรุปรีวิวนี้เป็น JSON: …" | ขอ JSON แต่ไม่บอกรูปแบบ ไม่มีตัวอย่าง |
| **C · เต็ม** | บทบาท + คำสั่งรูปแบบ + ตัวอย่าง + รีวิว | ครบ 4 ส่วน |

ทุกระดับวัดด้วย **ตัวชี้วัดชุดเดียวกัน**: จำนวน token · entropy ของ token แรก (กระจาย/เกาะกลุ่ม) · P(ขึ้นต้น JSON) · สัดส่วน attention sink · อัตรา parse ผ่าน — แล้วรวมเป็นตารางเดียวตอนท้าย

> **ก่อนรัน:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
!pip install -q -U "transformers>=4.51,<5" accelerate
import torch, json, re, math, collections
import pandas as pd
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, os, urllib.request
FONT = "Sarabun-Regular.ttf"                       # ฟอนต์ไทยสำหรับกราฟ (ฟอนต์เริ่มต้นของ Colab ไม่มีอักษรไทย)
if not os.path.exists(FONT):
    try: urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", FONT)
    except Exception as e: print("ดาวน์โหลดฟอนต์ไม่ได้ (กราฟจะไม่แสดงอักษรไทย):", e)
if os.path.exists(FONT):
    fm.fontManager.addfont(FONT); plt.rcParams["font.family"] = [fm.FontProperties(fname=FONT).get_name(), "DejaVu Sans"]   # ตัวหลังสำรองสำหรับสัญลักษณ์ที่ Sarabun ไม่มี
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings, transformers; warnings.filterwarnings("ignore"); transformers.utils.logging.set_verbosity_error()   # ซ่อน warning ที่ไม่เกี่ยวกับบทเรียน
MODEL = "scb10x/llama3.2-typhoon2-1b-instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
# attn_implementation="eager" จำเป็น เพื่อให้ดึง attention weights ออกมาดูได้ (ขั้น 2.4)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto", attn_implementation="eager").eval()
EOT = tok.convert_tokens_to_ids("<|eot_id|>")      # token จบคำตอบของ Llama 3 — ใช้บอก generate ว่าหยุดตรงไหน
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่พบ GPU — เปลี่ยน runtime เป็น T4")
print("โหลดเสร็จ:", model.config.num_hidden_layers, "ชั้น")

## 2.1 prompt 3 ระดับ — เก็บเป็น "ส่วน ๆ" เพื่อวัดรายส่วนได้
ทุก prompt เป็นรายการของ (ชื่อส่วน, ข้อความ) · ส่วนที่มีชื่อเดียวกันเทียบกันข้ามระดับได้ (เช่น "รีวิว" เป็นข้อความเดียวกันทุกระดับ)
ระดับ C มีครบ: บทบาท · คำสั่งรูปแบบ · ตัวอย่าง · คำสั่ง + รีวิว — 3 ส่วนแรกคือส่วนที่จะถอดทีละส่วนใน 2.6

In [ ]:
REVIEW  = "กาแฟดีแต่รอนานมาก"
SEGMENTS = ["บทบาท", "คำสั่งรูปแบบ", "ตัวอย่าง", "คำสั่ง", "รีวิว"]   # ลำดับคงที่ ใช้ทุกกราฟ

ROLE    = "คุณคือระบบวิเคราะห์รีวิวร้านกาแฟ"
FORMAT  = "ตอบเป็น JSON เท่านั้น ไม่ต้องอธิบาย มี field sentiment (บวก/ลบ/ผสม) และ issues (รายการประเด็นสั้น ๆ)"
EXAMPLE = 'ตัวอย่าง: รีวิว "ขนมอร่อย ที่จอดรถหายาก" → {"sentiment": "ผสม", "issues": ["ขนมอร่อย", "ที่จอดรถหายาก"]}'

def full_parts(role=True, fmt=True, example=True):
    parts = []
    if role:    parts.append(("บทบาท", ROLE))
    if fmt:     parts.append(("คำสั่งรูปแบบ", FORMAT))
    if example: parts.append(("ตัวอย่าง", EXAMPLE))
    parts.append(("คำสั่ง", "ตอนนี้วิเคราะห์รีวิวนี้:"))
    parts.append(("รีวิว", REVIEW))                       # ข้อความรีวิวแยกเป็นส่วนของตัวเอง เหมือนระดับ A/B
    return parts

PROMPTS = {
    "A คลุมเครือ": [("คำสั่ง", "สรุปรีวิวนี้หน่อย:"), ("รีวิว", REVIEW)],
    "B สั้น":      [("คำสั่ง", "สรุปรีวิวนี้เป็น JSON:"), ("รีวิว", REVIEW)],
    "C เต็ม":      full_parts(),
}

def to_messages(parts):
    # ส่วนทั้งหมด (รวม "บทบาท") ต่อกันเป็น user message เดียว คั่นด้วยขึ้นบรรทัดใหม่ (ระดับ A/B คั่นด้วยช่องว่าง)
    # — แล็บนี้ใส่บทบาทใน user แทนช่อง system เพื่อให้ทุกส่วนอยู่ในข้อความเดียว จะได้จับคู่ตำแหน่ง token กับส่วนได้ใน 2.4 (Lab 3–4 จะใช้ช่อง system ตามปกติ)
    sep = "\n" if len(parts) > 2 else " "
    return [{"role": "user", "content": sep.join(t for _, t in parts)}]

def render(msgs):
    # add_generation_prompt=True → เติมหัว assistant ให้ โมเดลจึงเริ่ม "ตอบ" ที่ token ถัดไป
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode(text):
    # template ใส่ <|begin_of_text|> มาแล้ว จึงไม่ต้องให้ tokenizer เติมซ้ำ
    return tok(text, return_tensors="pt", add_special_tokens=False, return_offsets_mapping=True)

for name, parts in PROMPTS.items():
    print(f"===== {name} =====")
    print(to_messages(parts)[0]["content"]); print()

print("===== ระดับ C หลังใส่ chat template (สิ่งที่โมเดลเห็นจริง) =====")
print(render(to_messages(PROMPTS["C เต็ม"])))

> template เติม system block (มีบรรทัดวันที่) และ token พิเศษให้เอง — นี่คือรูปแบบที่โมเดลถูกเทรนมาให้เห็น
> **ขั้น ①:** ทั้งสามระดับตอนนี้เป็นแค่ข้อความคนละความยาว โมเดลยังไม่รู้ว่าอันไหน "ดี"

## 2.2 ขั้น ② นับ token — ต้นทุนของ prompt แต่ละระดับ

In [ ]:
results = {}          # เก็บตัวชี้วัดทุกอย่างของทุกระดับ ไว้รวมเป็นตารางเดียวตอนท้าย
for name, parts in PROMPTS.items():
    enc = encode(render(to_messages(parts)))
    results[name] = {"token ทั้งหมด": enc.input_ids.shape[1]}
    print(f"{name:12s} {enc.input_ids.shape[1]:4d} token (รวม token ของ template)")

## 2.3 ขั้น ④–⑤ การกระจายของ token แรกของคำตอบ
รัน forward ครั้งเดียว (ยังไม่ generate) เอา **logits** (คะแนนดิบของทุก token) ที่ตำแหน่งสุดท้ายมา softmax แล้ววัด 2 อย่าง
- **entropy (bit)** — ตัวเลขเดียวที่บอกว่าการกระจาย "กระจาย" (สูง) หรือ "เกาะกลุ่ม" (ต่ำ) · entropy 0 = มั่นใจ token เดียว · entropy 3 ≈ ลังเลระหว่าง 8 token
- **P(ขึ้นต้น JSON)** — รวมความน่าจะเป็นของทุก token ที่ขึ้นต้นด้วย `{` หรือ ``` (มีหลายตัว ไม่ใช่แค่ `{`)

In [ ]:
@torch.no_grad()
def next_token_probs(parts):
    enc = encode(render(to_messages(parts)))
    out = model(input_ids=enc.input_ids.to(model.device))
    return torch.softmax(out.logits[0, -1].float(), dim=-1).cpu()   # logits ของตำแหน่งสุดท้าย → softmax

def entropy_bits(p):
    p = p[p > 0]; return float(-(p * torch.log2(p)).sum())

# กลุ่ม token ที่ "ขึ้นต้น JSON": "{", '{"', "```", "```json" ฯลฯ (สแกน vocab ครั้งเดียว ~10 วินาที)
json_start_ids = [i for i in range(len(tok)) if tok.decode([i]).lstrip().startswith(("{", "```"))]
print("จำนวน token ที่ขึ้นต้น JSON ได้:", len(json_start_ids))

probs = {}
for name, parts in PROMPTS.items():
    p = next_token_probs(parts); probs[name] = p
    results[name]["entropy (bit)"] = round(entropy_bits(p), 2)
    results[name]["P(ขึ้นต้น JSON)"] = round(p[json_start_ids].sum().item(), 3)
    top = torch.topk(p, 5)
    print(f"{name:12s} entropy {results[name]['entropy (bit)']:.2f} bit | P(JSON) {results[name]['P(ขึ้นต้น JSON)']:.3f} | top-5:",
          ", ".join(f"{tok.decode([i.item()])!r} {v:.2f}" for v, i in zip(top.values, top.indices)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (name, p) in zip(axes, probs.items()):
    top = torch.topk(p, 10)
    labels = [tok.decode([i.item()]).replace("\n", "⏎") or "␣" for i in top.indices]
    colors = ["#1D6F5B" if i.item() in set(json_start_ids) else "#9AA6B8" for i in top.indices]
    ax.bar(range(10), top.values.numpy(), color=colors)
    ax.set_xticks(range(10)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(0, 1); ax.set_title(f"{name}  (entropy {results[name]['entropy (bit)']:.2f} bit)")
axes[0].set_ylabel("ความน่าจะเป็นของ token แรก")
fig.suptitle("แท่งเขียว = token ที่ขึ้นต้น JSON · แกน y เท่ากันทั้งสามกราฟ เทียบความสูงได้ตรง ๆ"); plt.tight_layout(); plt.show()

**อ่านกราฟอย่างไร:** เทียบทั้งสามที่แกน y เดียวกัน — ระดับ A มักกระจาย (entropy สูง) และไม่มีแท่งเขียว เพราะไม่ได้ขอ JSON · ระดับ B ขอ JSON แล้วแต่ยังไม่บอกรูปแบบ มักได้ทั้ง `{` และ ``` ปนกัน · ระดับ C ควรเกาะกลุ่ม (entropy ต่ำ) ที่ token ขึ้นต้น JSON

> ถ้าผลไม่ตรงคาด ให้บันทึกไว้ — ค่าจริงสำคัญกว่าค่าที่คาด และจะได้ใช้อภิปรายใน 2.6

## 2.4 ขั้น ④ attention: ตำแหน่งสุดท้ายมองไปที่ส่วนไหนของ prompt
ถ้าดู attention ดิบ น้ำหนักจะกองที่ token แรก (`<|begin_of_text|>`) และ token พิเศษ — ปกติของโมเดลตระกูลนี้ (**attention sink**)
เราจึงตัด token พิเศษออก normalize ใหม่ เฉลี่ยทุกหัว แล้ว**รวมเป็นรายส่วน** โดยใช้ชื่อส่วนชุดเดียวกันทุกระดับ (ส่วนที่ระดับนั้นไม่มี = 0) จึงวางกราฟเทียบกันได้

In [ ]:
special_ids = set(tok.all_special_ids) | set(tok.get_added_vocab().values())

@torch.no_grad()
def attention_by_segment(parts, layer=8):
    text = render(to_messages(parts)); enc = encode(text)
    out = model(input_ids=enc.input_ids.to(model.device), output_attentions=True)
    att = out.attentions[layer][0].float().mean(0)[-1].cpu()      # ชั้น index 8 (จาก 0–15) เฉลี่ยทุกหัว (mean(0)) แล้วเอาแถวของ query ตัวสุดท้าย ([-1]) → น้ำหนักต่อ key
    spans = []
    for name, ptxt in parts:
        s = text.find(ptxt); spans.append((name, s, s + len(ptxt)))
    seg_w = {k: 0.0 for k in SEGMENTS + ["template"]}
    sink = 0.0
    for j, (tid, (a, b)) in enumerate(zip(enc.input_ids[0].tolist(), enc.offset_mapping[0].tolist())):   # (a, b) = ตำแหน่งตัวอักษรของ token
        if tid in special_ids: sink += att[j].item(); continue
        seg = "template"
        for name, s, e in spans:
            if a < e and b > s: seg = name; break   # token ที่ทับช่วงข้อความของส่วนไหน = ส่วนนั้น
        seg_w[seg] += att[j].item()
    tot = sum(seg_w.values())
    return {k: v / tot for k, v in seg_w.items()}, sink

# --- ดู "ดิบ" ก่อน (ระดับ C, 40 token แรก): น้ำหนักกองที่ token แรก ---
@torch.no_grad()
def raw_attention(parts, layer=8):
    enc = encode(render(to_messages(parts)))
    out = model(input_ids=enc.input_ids.to(model.device), output_attentions=True)
    return out.attentions[layer][0].float().mean(0)[-1].cpu(), enc.input_ids[0].tolist()
att_raw, ids_raw = raw_attention(PROMPTS["C เต็ม"])
k = min(40, len(ids_raw))
labels_raw = [tok.decode([i]).replace("\n", "⏎").replace(" ", "␣") or "∅" for i in ids_raw[:k]]   # ถอดแต่ละ token เป็นข้อความอ่านได้ (เศษไบต์ไทย = �)
plt.figure(figsize=(12, 3)); plt.bar(range(k), att_raw[:k].numpy(), color="#B03A2E")
plt.xticks(range(k), [t[:10] for t in labels_raw], rotation=70, fontsize=7)
plt.title("attention ดิบของตำแหน่งสุดท้าย (ระดับ C, ชั้น index 8) — น้ำหนักกองที่ token แรก = attention sink"); plt.tight_layout(); plt.show()
print("หมายเหตุ: 40 token แรกส่วนใหญ่เป็นของ template (หัว system + บรรทัดวันที่) ยังไม่ถึงข้อความของเรา — ดูแค่ว่าแท่งแรกสูงกว่าตัวอื่นมาก")

# --- รวมรายส่วน ทั้งสามระดับ บนแกนเดียวกัน ---
seg = {}
for name, parts in PROMPTS.items():
    w, sink = attention_by_segment(parts); seg[name] = w
    results[name]["attention sink"] = round(sink, 2)
    print(f"{name:12s} sink {sink:.2f} | " + "  ".join(f"{k} {v:.2f}" for k, v in w.items() if v > 0))   # แสดงเฉพาะส่วนที่ระดับนั้นมี

import numpy as np
x = np.arange(len(SEGMENTS) + 1); wdt = 0.26
plt.figure(figsize=(11, 3.6))
for i, (name, w) in enumerate(seg.items()):
    plt.bar(x + (i - 1) * wdt, [w[k] for k in SEGMENTS + ["template"]], wdt, label=name)
plt.xticks(x, SEGMENTS + ["template"]); plt.ylabel("สัดส่วน attention (ตัด token พิเศษแล้ว)")
plt.legend(); plt.title("token สุดท้ายมองไปที่ส่วนไหน — สามระดับบนแกนเดียวกัน"); plt.tight_layout(); plt.show()

**อ่านกราฟอย่างไร:** แท่ง "template" = ส่วนที่ chat template เติมให้เอง (system block ที่มีบรรทัดวันที่ + หัว user/assistant) ซึ่งมีในทุกระดับ · ระดับ A/B มีแค่ "คำสั่ง" กับ "รีวิว" ให้มอง ส่วนระดับ C มี "คำสั่งรูปแบบ" และ "ตัวอย่าง" เพิ่มเข้ามาให้ attention ดึงได้ · ค่า `sink` ในบรรทัดพิมพ์ = สัดส่วนที่กองอยู่ที่ token พิเศษ (ตัดทิ้งก่อนคิดสัดส่วน)

> ลองเปลี่ยน `layer=` เป็น 2, 8, 14 แล้วดูว่าสัดส่วนเปลี่ยนไหม — ชั้นต่างกันมอง prompt ต่างกัน
> **ย้ำ:** attention เป็นแค่ "ร่องรอย" ว่าโมเดลมองอะไร ไม่ใช่คำอธิบายว่าทำไมมันตอบแบบนั้น หลักฐานที่หนักแน่นกว่าคือ 2.5–2.6

## 2.5 ขั้น ⑥ generate จริง แล้ววัดอัตรา parse JSON ผ่าน
สุ่มที่ temperature เท่ากันทุกระดับ (กำหนด seed → รันซ้ำได้ผลเดิม) · ระดับ A ไม่ได้ขอ JSON จึงคาดว่า parse ไม่ผ่านอยู่แล้ว — ใส่ไว้เพื่อให้เห็นว่า "ไม่ระบุรูปแบบ" ให้ผลอย่างไร ไม่ใช่เพื่อตัดคะแนน

**parse ผ่าน** = ทั้งคำตอบเป็น JSON (ยอมให้ห่อด้วย ``` ได้) — ถ้ามีคำอธิบายปนก่อนหรือหลัง JSON ถือว่าไม่ผ่าน เพราะโปรแกรมที่รับคำตอบไปใช้ต่ออ่านไม่ได้

In [ ]:
N_SAMPLES = 5      # เพิ่มเป็น 10 ถ้ามีเวลา — 5 ครั้งยังผันผวนพอสมควร

def strip_fence(s):
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s); s = re.sub(r"\s*```$", "", s)
    return s.strip()

def try_parse(s):
    try: return json.loads(strip_fence(s))
    except Exception: return None

@torch.no_grad()
def generate(parts, n=N_SAMPLES, temperature=0.7, max_new_tokens=80):
    ids = tok(render(to_messages(parts)), return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    torch.manual_seed(0)      # สุ่มแบบกำหนด seed → รันซ้ำได้ผลเดิม และทุกระดับได้ "ลูกเต๋า" ชุดเดียวกัน
    outs = []
    for _ in range(n):
        g = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.9,
                           eos_token_id=EOT, pad_token_id=EOT)
        outs.append(tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True))
    return outs

samples = {}
for name, parts in PROMPTS.items():
    outs = generate(parts); samples[name] = outs
    ok = sum(try_parse(o) is not None for o in outs)
    results[name][f"parse ผ่าน/{N_SAMPLES}"] = ok
    print(f"\n=== {name}: parse ผ่าน {ok}/{N_SAMPLES} ===")
    for o in outs: print("  •", o.replace("\n", " ")[:110])

## ตารางสรุป: สามระดับ × ตัวชี้วัดทั้งหมด (token · entropy · P(JSON) · attention sink · parse ผ่าน)

In [ ]:
df = pd.DataFrame.from_dict(results, orient="index")     # แถว = ระดับ prompt, คอลัมน์ = ตัวชี้วัด
df

**วิธีอ่าน:** ไล่จาก A → C ควรเห็น token เพิ่ม (ต้นทุน), entropy ลด (เกาะกลุ่ม), P(JSON) เพิ่ม, parse ผ่านเพิ่ม · คอลัมน์ **attention sink** = สัดส่วนที่กองอยู่ที่ token พิเศษ มักสูงพอ ๆ กันทุกระดับ (เป็นนิสัยของโมเดล ไม่ได้บอกว่า prompt ดีหรือไม่ดี) · ถ้า B ใกล้ C อยู่แล้ว แปลว่าโจทย์นี้แค่ "ขอ JSON" ก็พอ ไม่ต้องจ่าย token เพิ่ม — นี่คือคำถามที่ Lab 3 จะวัดอย่างเป็นระบบ

## 2.6 ถอด prompt เต็มทีละส่วน — ส่วนไหน "ทำงาน" มากที่สุด?
เริ่มจากระดับ C แล้วเอาออกทีละส่วน (เหลืออีก 3 ส่วน) วัดด้วยตัวชี้วัดชุดเดิม · นี่คือหลักฐานที่หนักแน่นที่สุดของแล็บนี้ เพราะเปลี่ยนตัวแปรทีละอย่าง

In [ ]:
configs = [("− บทบาท", dict(role=False)), ("− คำสั่งรูปแบบ", dict(fmt=False)), ("− ตัวอย่าง", dict(example=False))]
abl = {"C ครบทุกส่วน": {k: v for k, v in results["C เต็ม"].items() if k != "attention sink"}}   # แถวอ้างอิง = ตัวเลขเดิมจากตารางข้างบน
for name, kw in configs:
    parts = full_parts(**kw)
    p = next_token_probs(parts)
    outs = generate(parts)
    abl[name] = {"token ทั้งหมด": encode(render(to_messages(parts))).input_ids.shape[1],
                 "entropy (bit)": round(entropy_bits(p), 2),
                 "P(ขึ้นต้น JSON)": round(p[json_start_ids].sum().item(), 3),
                 f"parse ผ่าน/{N_SAMPLES}": sum(try_parse(o) is not None for o in outs)}
pd.DataFrame.from_dict(abl, orient="index")

## ✍️ ข้อสรุปของคุณ (เขียน 3 บรรทัด)
1. ส่วนไหนของ prompt ที่พอถอดออกแล้ว P(JSON) และ parse ผ่านตกมากที่สุด? (มักเป็น "คำสั่งรูปแบบ" หรือ "ตัวอย่าง" — โจทย์นี้ต้องการ *รูปแบบ* ไม่ใช่ *น้ำเสียง* บทบาทจึงมักช่วยน้อย)
2. ผลใน 2.6 สอดคล้องกับ attention ใน 2.4 ไหม? ถ้าส่วนที่ attention มองมาก แต่ถอดแล้วผลไม่ตก แปลว่าอะไร?
3. เปลี่ยน `REVIEW` เป็นรีวิวของคุณเอง (ลองรีวิวประชด) แล้วรันใหม่ทั้งหมด — ระดับไหนยังทำงานได้?

## ✅ สรุป Lab 2
- prompt ทุกระดับคือ token ที่ต่อกันข้างหน้า ต่างกันที่ "ให้อะไรกับ attention ดึง" และต้นทุน token ที่จ่าย
- entropy ของ token แรกเป็นตัวเลขเดียวที่บอกว่า prompt ทำให้โมเดล "เกาะกลุ่ม" แค่ไหน — prompt ที่ระบุรูปแบบทำให้ entropy ต่ำ ผลจึงนิ่ง
- attention บอกได้ว่าโมเดล "มอง" อะไร · การถอดทีละส่วนบอกได้ว่าอะไร **มีผลจริง**